In [1]:
import os
from load_dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import AIMessage,HumanMessage,SystemMessage
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnableSequence, RunnableLambda

if os.environ.get("OPENAI_API_KEY"):
  print("API Key exists")
else:
  raise ValueError("No Key found")

llm=ChatOpenAI(model="gpt-4o-mini",
               temperature=0.9)


c:\Users\arunm\OneDrive\Desktop\LangChain_Tutorial\.venv\Lib\site-packages\langchain_core\_api\deprecation.py:25: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1.fields import FieldInfo as FieldInfoV1


API Key exists


In [2]:
imdb_prompt = ChatPromptTemplate.from_messages([
  ("system", "you are an IMDb movie reviewer"),
  ("human", "Write an IMDb-style review for the movie {movie}. Include a rating out of 10 and explain the strengths and weaknesses.")
])

rotten_tomatoes_prompt = ChatPromptTemplate.from_messages([
  ("system", "you are a Rotten Tomatoes movie reviewer"),
  ("human", "Write a Rotten Tomatoes-style review for the movie {movie}. Include a tomatometer-style score out of 100 and a concise critic-style opinion.")
])

In [3]:
llm = ChatOpenAI(model="gpt-4o-mini",
                 temperature=0.9)

In [4]:
str_parser = StrOutputParser()

In [5]:
imdb_chain = imdb_prompt | llm | str_parser

rotten_tomatoes_chain = rotten_tomatoes_prompt | llm | str_parser

In [6]:
parallel_reviews_chain = RunnableParallel(
  imdb=imdb_chain,
  rotten_tomatoes=rotten_tomatoes_chain,
 )

parallel_reviews_chain.invoke({"movie": "Inception"})

{'imdb': "**Title:** Inception  \n**Rating:** 9/10  \n\n**Review:**\n\nChristopher Nolan's *Inception* is nothing short of a cinematic masterpiece that beautifully intertwines the realms of dreams and reality. Released in 2010, this mind-bending thriller challenges viewers intellectually while delivering a visually stunning experience.\n\n**Strengths:**\n\nOne of the film's standout qualities is its intricate and original screenplay. Nolan introduces a compelling concept: dream manipulation and inception, where ideas can be planted within one's subconscious. The layers of dreams create an engaging narrative that keeps the audience on the edge of their seats. The film's structure, carefully orchestrated with multiple levels of dreams intersecting, showcases Nolan's exceptional storytelling ability.\n\nThe strong cast is another pillar of the film’s robust foundation. Leonardo DiCaprio delivers a poignant performance as Cobb, a skilled thief haunted by his past. His emotional struggle br

In [7]:
def beautify(response: dict) -> dict:
    return {
        "imdb": response["imdb"].strip(),
        "rotten_tomatoes": response["rotten_tomatoes"].strip(),
    }

beautify(parallel_reviews_chain.invoke({"movie": "Inception"}))

{'imdb': "**Inception (2010)**  \n**Rating: 9/10**\n\n**Review:**\n\nChristopher Nolan's *Inception* is a masterclass in science fiction storytelling that deftly blends intellectual depth with a thrilling heist narrative. This film not only challenges the boundaries of reality but also delves into the intricacies of the subconscious mind, making it a thought-provoking experience.\n\nOne of the film's notable strengths is its originality. The concept of dream manipulation and shared dreaming is not only fascinating but also opens up endless possibilities for narrative exploration. The screenplay, co-written by Nolan, is intricately layered, requiring viewers to pay close attention as it unfolds through its complex structure. The film's ability to balance action sequences with philosophical musings about identity and reality is commendable.\n\nVisually, *Inception* is a triumph. The special effects are groundbreaking, from the iconic bending of cityscapes to the awe-inspiring zero-gravit

In [8]:
from pydantic import BaseModel,Field
from typing import Literal

class llm_schema(BaseModel):
  movie_summary_flag: Literal["positive","negative"]

In [9]:
llm_structured_output = llm.with_structured_output(llm_schema)
llm_structured_output.invoke("movie is great")

c:\Users\arunm\OneDrive\Desktop\LangChain_Tutorial\.venv\Lib\site-packages\pydantic\main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=llm_schema(movie_summary_flag='positive'), input_type=llm_schema])
  return self.__pydantic_serializer__.to_python(


llm_schema(movie_summary_flag='positive')